# PaliGemma 2: bul, kes, tekrar oku

Bu deneyde PaliGemma 2 3B'nin mix-224 modeli önce fişteki toplam satırını `detect` komutuyla bulmaya çalıştı. Bulunan bölge orijinal fotoğraftan kesildi ve aynı modele tekrar okutuldu. Sonuçlar aynı fişlerde mix-448 modeliyle karşılaştırıldı.

| Yöntem | Ne yapıyor | Gemma'nın okuduğu resim notu |
|---|---|---|
| **A** | mix-224, bütün fiş | 256 |
| **B** | mix-448, bütün fiş | 1.024 |
| **C** | mix-224: `detect` ile bölgeyi bul → orijinal fotoğraftan kes → tekrar 224'te oku | 256 + 256 = 512 |
| **C\*** | Kontrol: toplam satırının kutusu elle verilir, kesilip 224'te okunur (bulma adımı devre dışı) | 256 |

Fiş fotoğrafları Wikimedia Commons'tan serbest lisanslıdır (6. hücre).

In [ ]:
# 0) Hugging Face girişi. Token Colab Secrets'tan (HF_TOKEN) okunur, kodda yazmaz.
#    Bu hücreyi Colab arayüzünden kendin çalıştır; Secrets uzaktan okunamaz.
from google.colab import userdata
from huggingface_hub import login
login(userdata.get("HF_TOKEN"))

In [ ]:
# 1) Kontrol: GPU, kütüphaneler ve Hugging Face erişimi (önce 0. hücreyle giriş yapılmış olmalı)
!nvidia-smi --query-gpu=name,memory.total --format=csv
import transformers, torch
print("transformers", transformers.__version__, "| torch", torch.__version__, "| cuda:", torch.cuda.is_available())

from huggingface_hub import model_info
for mid in ["google/paligemma2-3b-mix-224", "google/paligemma2-3b-mix-448"]:
    try:
        model_info(mid); print("erişim var:", mid)
    except Exception as e:
        print("ERİŞİM YOK:", mid, "->", type(e).__name__)

name, memory.total [MiB]
Tesla T4, 15360 MiB
transformers 5.20.1 | torch 2.11.0+cu130 | cuda: True
erişim var: google/paligemma2-3b-mix-224
erişim var: google/paligemma2-3b-mix-448


In [ ]:
# 2) Yardımcı fonksiyonlar
import re, os, gc, time, torch, numpy as np
from PIL import Image, ImageDraw
from transformers import PaliGemmaForConditionalGeneration, PaliGemmaProcessor

def load(model_id):
    proc = PaliGemmaProcessor.from_pretrained(model_id)
    model = PaliGemmaForConditionalGeneration.from_pretrained(
        model_id, dtype=torch.bfloat16, device_map="cuda").eval()
    return model, proc

def ask(model, proc, image, prompt, max_new_tokens=32):
    """Modele resim + metin ver, cevabı döndür. raw: özel token'lar dahil (detect çıktısı için)."""
    inputs = proc(text="<image>" + prompt, images=image.convert("RGB"), return_tensors="pt").to(model.device)
    inputs["pixel_values"] = inputs["pixel_values"].to(model.dtype)
    L = inputs["input_ids"].shape[-1]
    t0 = time.time()
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    gen = out[0][L:]
    return (proc.decode(gen, skip_special_tokens=True).strip(),
            proc.decode(gen, skip_special_tokens=False),
            round(time.time() - t0, 1))

def parse_box(raw, W, H):
    """'<loc0123><loc0456><loc0789><loc1000> total' -> (x0, y0, x1, y1) piksel. Yoksa None."""
    v = [int(x) for x in re.findall(r"<loc(\d{4})>", raw)]
    if len(v) < 4:
        return None
    y0, x0, y1, x1 = v[:4]
    return (x0 / 1024 * W, y0 / 1024 * H, x1 / 1024 * W, y1 / 1024 * H)

def square_crop(img, box, pad=0.15):
    """Kutunun etrafından kare bir bölge kes, taşan yeri beyazla doldur."""
    W, H = img.size
    x0, y0, x1, y1 = box
    cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
    side = max(max(x1 - x0, y1 - y0) * (1 + 2 * pad), 0.08 * min(W, H))
    l, t, s = int(cx - side / 2), int(cy - side / 2), int(side)
    canvas = Image.new("RGB", (s, s), "white")
    canvas.paste(img.crop((max(0, l), max(0, t), min(W, l + s), min(H, t + s))), (max(0, -l), max(0, -t)))
    return canvas

def to_number(s):
    s = s.replace(" ", "")
    if "," in s and "." in s:
        s = s.replace(".", "").replace(",", ".") if s.rfind(",") > s.rfind(".") else s.replace(",", "")
    elif "," in s:
        s = s.replace(",", ".") if len(s.split(",")[-1]) in (1, 2) else s.replace(",", "")
    elif s.count(".") > 1:
        head, tail = s.rsplit(".", 1); s = head.replace(".", "") + "." + tail
    return float(s)

def is_correct(answer, truth):
    t = to_number(truth)
    for m in re.findall(r"\d[\d.,]*\d|\d", answer):
        try:
            if abs(to_number(m) - t) < 0.011:
                return True
        except ValueError:
            pass
    return False

print("hazır")

hazır


In [ ]:
# 3) Deneme: 224 modelini yükle, yapay bir fişle her şeyin çalıştığını kontrol et
from PIL import ImageFont
def fake_receipt():
    img = Image.new("RGB", (600, 1400), "white"); d = ImageDraw.Draw(img)
    try: f = ImageFont.truetype("/usr/share/fonts/truetype/liberation/LiberationMono-Regular.ttf", 28)
    except: f = ImageFont.load_default()
    lines = ["   MARKET ORNEK", "", "SUT 1L          34.50", "EKMEK           12.00", "PEYNIR         145.90",
             "ELMA 1KG        38.75", "DETERJAN       118.85", "SU 5L           22.40", "CIKOLATA        29.90",
             "", "ARA TOPLAM     402.30", "KDV             36.57", "", "TOPLAM         438.87", "", "KART ILE ODENDI", "TESEKKURLER"]
    for i, l in enumerate(lines): d.text((40, 60 + i * 60), l, fill="black", font=f)
    return img
test = fake_receipt()

m224, p224 = load("google/paligemma2-3b-mix-224")
print("GPU bellek:", round(torch.cuda.memory_allocated() / 1e9, 1), "GB")

q = "answer en What is the total amount?"
a, _, t = ask(m224, p224, test, q); print(f"A (224, bütün fiş): {a!r}  {t}s  doğru mu: {is_correct(a, '438.87')}")
_, raw, t = ask(m224, p224, test, "detect total"); print(f"detect çıktısı: {raw!r}  {t}s")
box = parse_box(raw, *test.size); print("kutu:", box)
if box:
    crop = square_crop(test, box)
    c, _, t = ask(m224, p224, crop, q); print(f"C (kes + 224): {c!r}  {t}s  doğru mu: {is_correct(c, '438.87')}")

preprocessor_config.json:   0%|          | 0.00/424 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/243k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 34.6MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/727 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

GPU bellek: 6.1 GB
A (224, bütün fiş): '668.87'  2.5s  doğru mu: False
detect çıktısı: '<loc0000><loc0000><loc1008><loc1023> total<eos>'  0.8s
kutu: (0.0, 0.0, 599.4140625, 1378.125)
C (kes + 224): '410.67'  0.8s  doğru mu: False


In [ ]:
# 4) Hangi detect komutu toplam satırını buluyor? (yapay fiş, doğru satır y≈840-870 px)
for p in ["detect total", "detect total amount", "detect price", "detect number", "detect text", "detect receipt total", "detect word"]:
    _, raw, t = ask(m224, p224, test, p, max_new_tokens=64)
    boxes = re.findall(r"((?:<loc\d{4}>){4})\s*([^<;]*)", raw)
    out = []
    for b, lab in boxes[:6]:
        v = [int(x) for x in re.findall(r"\d{4}", b)]
        out.append(f"y {v[0]/1024*1400:.0f}-{v[2]/1024*1400:.0f} x {v[1]/1024*600:.0f}-{v[3]/1024*600:.0f} '{lab.strip()}'")
    print(f"{p:22s} -> {out if out else raw!r}")

detect total           -> ["y 0-1378 x 0-599 'total'"]
detect total amount    -> ["y 0-1378 x 0-599 'total amount'"]
detect price           -> '<eos>'
detect number          -> '<eos>'
detect text            -> '<eos>'
detect receipt total   -> ["y 0-1378 x 0-599 'receipt total'"]
detect word            -> '<eos>'


In [ ]:
# 5) Fiş fotoğrafın küçük bir kısmıysa: "detect receipt" ile fişi bulup kesmek işe yarıyor mu?
scene = Image.new("RGB", (1800, 1350), (120, 92, 70))                 # masa
scene.paste(test.resize((300, 700)), (980, 420))                     # fiş, karenin küçük bir kısmı
q = "answer en What is the total amount?"
a, _, t = ask(m224, p224, scene, q); print(f"A (224, bütün sahne): {a!r} doğru mu: {is_correct(a, '438.87')}")
for p in ["detect receipt", "detect paper", "detect document"]:
    _, raw, _ = ask(m224, p224, scene, p)
    box = parse_box(raw, *scene.size); print(f"{p:16s} kutu: {tuple(round(v) for v in box) if box else raw!r}")
    if box:
        c, _, _ = ask(m224, p224, square_crop(scene, box, pad=0.05), q)
        print(f"   C (kes + 224): {c!r} doğru mu: {is_correct(c, '438.87')}")

A (224, bütün sahne): '56.89' doğru mu: False
detect receipt   kutu: '<eos>'
detect paper     kutu: '<eos>'
detect document  kutu: '<eos>'


In [ ]:
# 6) Fişler: Wikimedia Commons, serbest lisanslı. Doğru toplamlar tam çözünürlükte elle kontrol edildi.
import urllib.request, urllib.parse, json, io, time
TITLES = {
 "1_alexanders":   ("File:Alexander's Supermarket receipt, late twentieth century.tif", "39.99", "CC0 · Struthious Bandersnatch · Wikimedia Commons"),
 "2_mcdonalds_hk": ("File:HK Central McDonalds Lunch set receipt Dec-2010.jpg",        "29.30", "CC BY-SA 3.0 · PeterLigerry · Wikimedia Commons"),
 "3_biedronka":    ("File:Polish supermarket receipt.jpg",                             "8,27",  "CC BY-SA 4.0 · Hippietrail · Wikimedia Commons"),
 "4_shell":        ("File:Shell-Gas-Station-Receipt-MasterCard.jpg",                   "20.00", "Public domain · Wikimedia Commons"),
 "5_tesco":        ("File:Tesco grocery receipt Finchley 1994.jpg",                    "6.71",  "Public domain · Wikimedia Commons"),
}
UA = {"User-Agent": "PaliGemmaReceiptExperiment/1.0 (personal educational project)"}
q = "https://commons.wikimedia.org/w/api.php?" + urllib.parse.urlencode(
    {"action": "query", "titles": "|".join(t[0] for t in TITLES.values()), "prop": "imageinfo", "iiprop": "url", "format": "json"})
urls = {p["title"]: p["imageinfo"][0]["url"]
        for p in json.load(urllib.request.urlopen(urllib.request.Request(q, headers=UA)))["query"]["pages"].values()}
os.makedirs("/content/receipts", exist_ok=True)
DATA = {}
for key, (title, total, lic) in TITLES.items():
    fn = f"/content/receipts/{key}.png"
    for attempt in range(5):
        try:
            raw = urllib.request.urlopen(urllib.request.Request(urls[title], headers=UA), timeout=60).read(); break
        except Exception as e:
            print(key, "tekrar deneniyor:", e); time.sleep(15)
    Image.open(io.BytesIO(raw)).convert("RGB").save(fn)
    DATA[key] = dict(path=fn, total=total, lic=lic)
    print(key, Image.open(fn).size, "| doğru toplam:", total, "|", lic)
    time.sleep(5)

1_alexanders (768, 1552) | doğru toplam: 39.99 | CC0 · Struthious Bandersnatch · Wikimedia Commons
2_mcdonalds_hk (2592, 1936) | doğru toplam: 29.30 | CC BY-SA 3.0 · PeterLigerry · Wikimedia Commons
3_biedronka (2856, 3879) | doğru toplam: 8,27 | CC BY-SA 4.0 · Hippietrail · Wikimedia Commons
4_shell (915, 1415) | doğru toplam: 20.00 | Public domain · Wikimedia Commons
5_tesco (769, 2060) | doğru toplam: 6.71 | Public domain · Wikimedia Commons


In [ ]:
# 7) A ve C yöntemleri (mix-224)
os.makedirs("/content/out", exist_ok=True)
Q = "answer en What is the total amount?"
rows = []
for key, d in DATA.items():
    img = Image.open(d["path"]).convert("RGB"); W, H = img.size
    a, _, t = ask(m224, p224, img, Q)
    rows.append(dict(fis=key, yontem="A: 224 bütün fiş", cevap=a, dogru=is_correct(a, d["total"]), saniye=t, resim_notu=256))
    for target in ["total", "receipt"]:
        _, raw, _ = ask(m224, p224, img, f"detect {target}")
        box = parse_box(raw, W, H)
        whole = box is not None and (box[2]-box[0]) * (box[3]-box[1]) > 0.9 * W * H
        if box is None:
            rows.append(dict(fis=key, yontem=f"C: detect {target}", cevap="(kutu bulunamadı)", dogru=False, saniye=None, resim_notu=512)); continue
        crop = square_crop(img, box, pad=0.05 if target == "receipt" else 0.15)
        crop.save(f"/content/out/{key}_crop_{target}.png")
        vis = img.copy(); ImageDraw.Draw(vis).rectangle(box, outline=(255, 140, 0), width=max(4, W // 150)); vis.save(f"/content/out/{key}_box_{target}.png")
        c, _, t = ask(m224, p224, crop, Q)
        rows.append(dict(fis=key, yontem=f"C: detect {target}", cevap=c + ("  [kutu = bütün resim]" if whole else ""),
                         dogru=is_correct(c, d["total"]), saniye=t, resim_notu=512,
                         kutu=tuple(round(v) for v in box)))
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 60)
print(pd.DataFrame(rows)[["fis", "yontem", "cevap", "dogru", "kutu"]].to_string(index=False))

           fis            yontem                         cevap  dogru                kutu
  1_alexanders  A: 224 bütün fiş                         37.85  False                 NaN
  1_alexanders   C: detect total             (kutu bulunamadı)  False                 NaN
  1_alexanders C: detect receipt             (kutu bulunamadı)  False                 NaN
2_mcdonalds_hk  A: 224 bütün fiş                         13.70  False                 NaN
2_mcdonalds_hk   C: detect total             (kutu bulunamadı)  False                 NaN
2_mcdonalds_hk C: detect receipt             (kutu bulunamadı)  False                 NaN
   3_biedronka  A: 224 bütün fiş                          8.27   True                 NaN
   3_biedronka   C: detect total    8.27  [kutu = bütün resim]   True  (0, 0, 2850, 3871)
   3_biedronka C: detect receipt             (kutu bulunamadı)  False                 NaN
       4_shell  A: 224 bütün fiş                       $ 28.89  False                 NaN
       4_s

In [ ]:
# 8) B yöntemi (mix-448, bütün fiş). 224 modeli bellekten çıkarılıyor.
del m224, p224; gc.collect(); torch.cuda.empty_cache()
m448, p448 = load("google/paligemma2-3b-mix-448")
for key, d in DATA.items():
    img = Image.open(d["path"]).convert("RGB")
    b, _, t = ask(m448, p448, img, Q)
    rows.append(dict(fis=key, yontem="B: 448 bütün fiş", cevap=b, dogru=is_correct(b, d["total"]), saniye=t, resim_notu=1024))
df = pd.DataFrame(rows)
tab = df[df.yontem.isin(["A: 224 bütün fiş", "B: 448 bütün fiş"])].pivot(index="fis", columns="yontem", values="cevap")
tab.insert(0, "dogru_toplam", [DATA[k]["total"] for k in tab.index])
print(tab.to_string())
print()
print(df.groupby("yontem")[["dogru"]].sum().rename(columns={"dogru": "dogru_sayisi (5 fiş)"}).to_string())

preprocessor_config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.36k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/243k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/733 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/75.1k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/727 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

yontem         dogru_toplam A: 224 bütün fiş B: 448 bütün fiş
fis                                                          
1_alexanders          39.99            37.85            39.85
2_mcdonalds_hk        29.30            13.70            23.80
3_biedronka            8,27             8.27             8.27
4_shell               20.00          $ 28.89            20.00
5_tesco                6.71             4.71             6.71

                   dogru_sayisi (5 fiş)
yontem                                 
A: 224 bütün fiş                      1
B: 448 bütün fiş                      3
C: detect receipt                     0
C: detect total                       1


In [ ]:
# 9) Kontrol deneyi: Bölge DOĞRU bulunsaydı, kesip 224'te okumak işe yarar mıydı?
#    Toplam satırının kutusu elle verildi (detect adımını devre dışı bırakıp sadece "yaklaşma" kısmını ölçmek için).
del m448, p448; gc.collect(); torch.cuda.empty_cache()
m224, p224 = load("google/paligemma2-3b-mix-224")
MANUAL_BOX = {   # (x0, y0, x1, y1) piksel, tam çözünürlüklü orijinal fotoğrafta toplam satırı
  "1_alexanders":   (120, 1245, 640, 1305),
  "2_mcdonalds_hk": (1100, 1100, 1270, 1830),   # fiş yan dönmüş, satır dikey
  "3_biedronka":    (0, 2380, 2856, 2620),
  "4_shell":        (0, 990, 915, 1120),
  "5_tesco":        (80, 1320, 710, 1390),
}
for key, d in DATA.items():
    img = Image.open(d["path"]).convert("RGB")
    crop = square_crop(img, MANUAL_BOX[key], pad=0.05)
    crop.save(f"/content/out/{key}_crop_elle.png")
    vis = img.copy(); ImageDraw.Draw(vis).rectangle(MANUAL_BOX[key], outline=(255, 140, 0), width=max(4, img.width // 150)); vis.save(f"/content/out/{key}_box_elle.png")
    c, _, t = ask(m224, p224, crop, Q)
    rows.append(dict(fis=key, yontem="C*: elle kes + 224", cevap=c, dogru=is_correct(c, d["total"]), saniye=t, resim_notu=256,
                     kutu=MANUAL_BOX[key]))
df = pd.DataFrame(rows)
order = ["A: 224 bütün fiş", "B: 448 bütün fiş", "C: detect total", "C: detect receipt", "C*: elle kes + 224"]
tab = df.pivot(index="fis", columns="yontem", values="cevap")[order]
tab.insert(0, "dogru_toplam", [DATA[k]["total"] for k in tab.index])
print(tab.to_string()); print()
print(df.groupby("yontem")["dogru"].sum().reindex(order).rename("dogru (5 fişte)").to_string())

Loading weights:   0%|          | 0/727 [00:00<?, ?it/s]

yontem         dogru_toplam A: 224 bütün fiş B: 448 bütün fiş                C: detect total  C: detect receipt C*: elle kes + 224
fis                                                                                                                               
1_alexanders          39.99            37.85            39.85              (kutu bulunamadı)  (kutu bulunamadı)              48.88
2_mcdonalds_hk        29.30            13.70            23.80              (kutu bulunamadı)  (kutu bulunamadı)              23.80
3_biedronka            8,27             8.27             8.27     8.27  [kutu = bütün resim]  (kutu bulunamadı)               8.27
4_shell               20.00          $ 28.89            20.00  $ 28.00  [kutu = bütün resim]  (kutu bulunamadı)             $28.88
5_tesco                6.71             4.71             6.71                          16.70  (kutu bulunamadı)               6.71

yontem
A: 224 bütün fiş      1
B: 448 bütün fiş      3
C: detect total       1
C: 

In [ ]:
# 10) Kesitin şekli sonucu değiştiriyor mu? (aynı elle verilen kutular, 224 model)
#   kare+boşluk : kutu kareye tamamlanıp beyazla dolduruluyor (9. hücre)
#   gerdir      : kutu olduğu gibi kesiliyor; model resmi zaten 224x224'e gerdirerek alıyor
#   bağlamlı    : kutu dikeyde genişletiliyor (üst/alt satırlar da giriyor), kareye yakın oluyor
def stretch_crop(img, box):
    return img.crop(tuple(int(v) for v in box))
def context_crop(img, box):
    W, H = img.size; x0, y0, x1, y1 = box; w, h = x1 - x0, y1 - y0
    if w > h:   # yatay satır: yüksekliği genişliğin yarısına çıkar
        cy = (y0 + y1) / 2; nh = max(h, w * 0.5); y0, y1 = max(0, cy - nh / 2), min(H, cy + nh / 2)
    else:       # dikey satır (yan dönmüş fiş): genişliği yüksekliğin yarısına çıkar
        cx = (x0 + x1) / 2; nw = max(w, h * 0.5); x0, x1 = max(0, cx - nw / 2), min(W, cx + nw / 2)
    return img.crop((int(x0), int(y0), int(x1), int(y1)))
for key, d in DATA.items():
    img = Image.open(d["path"]).convert("RGB")
    for name, fn in [("C*: elle kes, gerdir", stretch_crop), ("C*: elle kes, bağlamlı", context_crop)]:
        crop = fn(img, MANUAL_BOX[key]); crop.save(f"/content/out/{key}_{name.split(', ')[1]}.png")
        c, _, t = ask(m224, p224, crop, Q)
        rows.append(dict(fis=key, yontem=name, cevap=c, dogru=is_correct(c, d["total"]), saniye=t, resim_notu=256, kutu=MANUAL_BOX[key]))
df = pd.DataFrame(rows)
cols = ["C*: elle kes + 224", "C*: elle kes, gerdir", "C*: elle kes, bağlamlı"]
tab = df[df.yontem.isin(cols)].pivot(index="fis", columns="yontem", values="cevap")[cols]
tab.insert(0, "dogru_toplam", [DATA[k]["total"] for k in tab.index])
print(tab.to_string()); print()
print(df.groupby("yontem")["dogru"].sum().reindex(order + cols[1:]).rename("dogru (5 fişte)").to_string())

yontem         dogru_toplam C*: elle kes + 224 C*: elle kes, gerdir C*: elle kes, bağlamlı
fis                                                                                       
1_alexanders          39.99              48.88                33.99                  39.85
2_mcdonalds_hk        29.30              23.80                46.50                  23.30
3_biedronka            8,27               8.27                $8.27                   8.27
4_shell               20.00             $28.88               $20.00                 $20.00
5_tesco                6.71               6.71                 6.71                   6.71

yontem
A: 224 bütün fiş          1
B: 448 bütün fiş          3
C: detect total           1
C: detect receipt         0
C*: elle kes + 224        2
C*: elle kes, gerdir      3
C*: elle kes, bağlamlı    3


In [ ]:
# 11) Paket: sonuç tablosu (CSV), özet grafiği, örnek görseller, lisans/kaynak dosyası -> /content/out.zip
import matplotlib.pyplot as plt, shutil
df.to_csv("/content/out/sonuclar.csv", index=False)

# Özet grafiği: yöntem başına doğru sayısı ve resim notu
summary = [("A: 224\nbütün fiş", 256), ("B: 448\nbütün fiş", 1024), ("C: detect total\n+ kes + 224", 512), ("C*: elle kes\n(bağlamlı) + 224", 256)]
keys = ["A: 224 bütün fiş", "B: 448 bütün fiş", "C: detect total", "C*: elle kes, bağlamlı"]
correct = [int(df[df.yontem == k].dogru.sum()) for k in keys]
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar([s[0] for s in summary], correct, color=["#c58a12", "#7a5fc4", "#9aa4bb", "#5c9a4b"])
for b, (lab, tok), c in zip(bars, summary, correct):
    ax.text(b.get_x() + b.get_width() / 2, c + 0.08, f"{c}/5\n{tok} resim notu", ha="center", va="bottom", fontsize=9)
ax.set_ylim(0, 5); ax.set_ylabel("doğru okunan toplam (5 fişte)")
ax.set_title("Fişteki toplamı okuma: 5 serbest lisanslı fiş, PaliGemma 2 3B mix")
ax.spines[["top", "right"]].set_visible(False); fig.tight_layout(); fig.savefig("/content/out/ozet_grafik.png", dpi=200); plt.close(fig)

# Örnek görsel: her fiş için (kutulu orijinal | bağlamlı kesit) + cevaplar
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
for i, (key, d) in enumerate(DATA.items()):
    axes[0, i].imshow(Image.open(f"/content/out/{key}_box_elle.png")); axes[0, i].set_title(key, fontsize=9)
    axes[1, i].imshow(Image.open(f"/content/out/{key}_bağlamlı.png"))
    r = df[(df.fis == key)].set_index("yontem").cevap
    axes[1, i].set_title(f"doğru: {d['total']}\n224: {r['A: 224 bütün fiş']} | 448: {r['B: 448 bütün fiş']}\nkes+224: {r['C*: elle kes, bağlamlı']}", fontsize=8)
    for a in axes[:, i]: a.axis("off")
fig.suptitle("Üst: toplam satırı (turuncu kutu). Alt: o bölgenin kesilip 224 modele verilen hali.", fontsize=10)
fig.tight_layout(); fig.savefig("/content/out/ornekler.png", dpi=150); plt.close(fig)

with open("/content/out/KAYNAKLAR_LISANS.txt", "w", encoding="utf-8") as f:
    f.write("Fiş fotoğrafları Wikimedia Commons'tan alınmıştır:\n")
    for key, (title, total, lic) in TITLES.items():
        f.write(f"- {key}: {title} | {lic} | https://commons.wikimedia.org/wiki/{urllib.parse.quote(title.replace(' ', '_'))}\n")
    f.write("\nCC BY-SA görseller (2 ve 3) kullanılırken fotoğrafçı ve lisans belirtilmelidir.\n")
    f.write("Model: google/paligemma2-3b-mix-224 ve mix-448 (Gemma Terms of Use). Donanım: Colab T4.\n")
    f.write("C* satırlarında toplam satırının kutusu elle verilmiştir (detect adımı devre dışı).\n")
shutil.make_archive("/content/sonuclar_paligemma_fis", "zip", "/content/out")
print(sorted(os.listdir("/content/out"))); print("zip:", round(os.path.getsize("/content/sonuclar_paligemma_fis.zip") / 1e6, 1), "MB")

['1_alexanders_bağlamlı.png', '1_alexanders_box_elle.png', '1_alexanders_crop_elle.png', '1_alexanders_gerdir.png', '2_mcdonalds_hk_bağlamlı.png', '2_mcdonalds_hk_box_elle.png', '2_mcdonalds_hk_crop_elle.png', '2_mcdonalds_hk_gerdir.png', '3_biedronka_bağlamlı.png', '3_biedronka_box_elle.png', '3_biedronka_box_total.png', '3_biedronka_crop_elle.png', '3_biedronka_crop_total.png', '3_biedronka_gerdir.png', '4_shell_bağlamlı.png', '4_shell_box_elle.png', '4_shell_box_total.png', '4_shell_crop_elle.png', '4_shell_crop_total.png', '4_shell_gerdir.png', '5_tesco_bağlamlı.png', '5_tesco_box_elle.png', '5_tesco_box_total.png', '5_tesco_crop_elle.png', '5_tesco_crop_total.png', '5_tesco_gerdir.png', 'KAYNAKLAR_LISANS.txt', 'ornekler.png', 'ozet_grafik.png', 'sonuclar.csv']
zip: 36.2 MB
